In [ ]:
#
import os
import re
import json
import copy
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

# ================= CONFIG =================
LABEL_CSV = "../LiverFibrosisCohort/LiverFibrosisCohort_MasterTable_Proccessed_527.csv"

RAD_ROOT = "/mnt/12T_Ironwolf/LiverData/Radiomics"
DEEP_ROOT = "../LiverFibrosisCohort/deep_medsiglip_v2"

# requested order
MODALITIES = ["T1", "T2", "ADC"]
DEV_SITES = ["CCHMC", "WISC","MICH"]

ID_COLUMN = "Image ID"
LABEL_COLUMN = "Fibrosis_Label"

RAD_HIDDEN_SIZES = [32, 16, 8]
DEEP_HIDDEN_SIZES = [128, 64, 32, 16]

EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 5e-5
DROPOUT = 0.10
PATIENCE = 15
MIN_DELTA = 1e-3
INNER_VAL_FRAC = 0.20
N_SPLITS = 5
SEED = 42
MAX_RAD_FEATURES = 16
THRESH_GRID = np.arange(0.4, 0.6, 0.02)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = "../Results/fusion_rad_deep_results_mean_impute"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ================= REPRODUCIBILITY =================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

# ================= ID HELPERS =================
def normalize_id(pid):
    pid = str(pid)

    m = re.match(r"PT-(\d{3}-\d{4})-", pid)
    if m:
        return m.group(1)

    m = re.match(r"PT-(M\d+)-", pid)
    if m:
        return m.group(1)

    return pid


def get_site(pid):
    pid = str(pid).strip().upper()

    if re.match(r"^\d{3}-\d{4}$", pid):
        return "CCHMC"
    elif pid.startswith("GJ"):
        return "WISC"
    elif pid.startswith("C"):
        return "MICH"
    elif re.match(r"^\d+$", pid):
        return "NYU"
    else:
        return "UNKNOWN"


# ================= LABEL =================
def binarize_fibrosis(x):
    if pd.isna(x):
        return np.nan

    x = str(x)
    m = re.search(r"\d+", x)
    if not m:
        return np.nan

    val = int(m.group())

    if val in [0, 1]:
        return 0
    elif val in [2, 3, 4]:
        return 1
    return np.nan


# ================= METRICS =================
def compute_sen_spe(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sen = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spe = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return sen, spe


def safe_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def tune_threshold(y_true, y_prob, grid=THRESH_GRID):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    if len(np.unique(y_true)) < 2:
        return 0.5, np.nan

    best_thr = 0.5
    best_bal = -np.inf

    for thr in grid:
        pred = (y_prob >= thr).astype(int)
        bal = balanced_accuracy_score(y_true, pred)
        if bal > best_bal:
            best_bal = bal
            best_thr = thr

    return best_thr, best_bal


# ================= DATA LOADING =================
def load_labels():
    labels_df = pd.read_csv(LABEL_CSV)
    labels_df = labels_df.rename(columns={ID_COLUMN: "id"})

    labels_df["id"] = labels_df["id"].astype(str).apply(normalize_id)
    labels_df["label"] = labels_df[LABEL_COLUMN].apply(binarize_fibrosis)

    labels_df = labels_df.dropna(subset=["label"]).copy()
    labels_df["label"] = labels_df["label"].astype(int)
    labels_df["site"] = labels_df["id"].apply(get_site)

    labels_df = labels_df[["id", "label", "site"]]
    labels_df = labels_df[labels_df["site"].isin(DEV_SITES)].copy()

    return labels_df


def load_radiomics(modality):
    liver_path = os.path.join(RAD_ROOT, f"{modality}_liver_features.csv")
    spleen_path = os.path.join(RAD_ROOT, f"{modality}_spleen_features.csv")

    if not os.path.exists(liver_path):
        raise FileNotFoundError(f"Missing file: {liver_path}")
    if not os.path.exists(spleen_path):
        raise FileNotFoundError(f"Missing file: {spleen_path}")

    liver_df = pd.read_csv(liver_path)
    spleen_df = pd.read_csv(spleen_path)

    liver_df["ID"] = liver_df["ID"].astype(str).apply(normalize_id)
    spleen_df["ID"] = spleen_df["ID"].astype(str).apply(normalize_id)

    liver_df = liver_df.rename(columns={"ID": "id"})
    spleen_df = spleen_df.rename(columns={"ID": "id"})

    rad_df = liver_df.merge(spleen_df, on="id", how="outer", suffixes=("_liver", "_spleen"))
    return rad_df


def load_deep(modality):
    feat_path = os.path.join(DEEP_ROOT, f"{modality}_medsiglip.csv")
    if not os.path.exists(feat_path):
        raise FileNotFoundError(f"Missing file: {feat_path}")

    deep_df = pd.read_csv(feat_path)
    deep_df["id"] = deep_df["id"].astype(str).apply(normalize_id)
    return deep_df


# ================= FEATURE PROCESSING =================
class RadiomicsPreprocessor:
    """
    Fold-specific preprocessing for radiomics:
    1) mean imputation
    2) remove zero-variance columns safely
    3) SelectKBest safely
    4) standardization
    """
    def __init__(self, max_features=64):
        self.max_features = max_features
        self.imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
        self.scaler = None
        self.var_keep_idx = None
        self.kbest = None
        self.use_fallback_zero = False
        self._n_selected = 0

    def fit(self, X, y):
        X = self.imputer.fit_transform(X)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        if X.shape[1] == 0:
            self.use_fallback_zero = True
            self.scaler = StandardScaler()
            self.scaler.fit(np.zeros((X.shape[0], 1), dtype=np.float32))
            self._n_selected = 1
            return self

        variances = np.var(X, axis=0)
        self.var_keep_idx = np.where(variances > 0)[0]

        if len(self.var_keep_idx) == 0:
            self.use_fallback_zero = True
            self.scaler = StandardScaler()
            self.scaler.fit(np.zeros((X.shape[0], 1), dtype=np.float32))
            self._n_selected = 1
            return self

        X = X[:, self.var_keep_idx]

        n_feats = X.shape[1]
        k = min(self.max_features, n_feats)

        if k > 0 and n_feats > k:
            self.kbest = SelectKBest(score_func=f_classif, k=k)
            X = self.kbest.fit_transform(X, y)
        else:
            self.kbest = None

        if X.shape[1] == 0:
            self.use_fallback_zero = True
            self.scaler = StandardScaler()
            self.scaler.fit(np.zeros((X.shape[0], 1), dtype=np.float32))
            self._n_selected = 1
            return self

        self.scaler = StandardScaler()
        self.scaler.fit(X)
        self._n_selected = X.shape[1]
        return self

    def transform(self, X):
        X = self.imputer.transform(X)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        if self.use_fallback_zero:
            X = np.zeros((X.shape[0], 1), dtype=np.float32)
            return self.scaler.transform(X)

        X = X[:, self.var_keep_idx]

        if self.kbest is not None:
            X = self.kbest.transform(X)

        X = self.scaler.transform(X)
        return X

    def fit_transform(self, X, y):
        self.fit(X, y)
        return self.transform(X)

    def n_selected(self):
        return int(self._n_selected)


class DeepPreprocessor:
    """
    Fold-specific preprocessing for deep features:
    1) mean imputation
    2) standardization
    """
    def __init__(self):
        self.imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
        self.scaler = None
        self.use_fallback_zero = False
        self._n_features = 0

    def fit(self, X):
        X = self.imputer.fit_transform(X)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        if X.shape[1] == 0:
            self.use_fallback_zero = True
            self.scaler = StandardScaler()
            self.scaler.fit(np.zeros((X.shape[0], 1), dtype=np.float32))
            self._n_features = 1
            return self

        self.scaler = StandardScaler()
        self.scaler.fit(X)
        self._n_features = X.shape[1]
        return self

    def transform(self, X):
        X = self.imputer.transform(X)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        if self.use_fallback_zero:
            X = np.zeros((X.shape[0], 1), dtype=np.float32)
            return self.scaler.transform(X)

        X = self.scaler.transform(X)
        return X

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    def n_features_used(self):
        return int(self._n_features)


# ================= MODEL =================
class FusionMLP(nn.Module):
    """
    Hidden design unchanged:
      rad:  Linear(rad_dim -> rad_hidden) + ReLU
      deep: Linear(deep_dim -> deep_hidden) + ReLU
      clf:  Linear(rad_hidden + deep_hidden -> 1)

    Added:
      - dropout after hidden activations
      - learnable per-unit gates
    """
    def __init__(self, rad_dim, deep_dim, rad_hidden, deep_hidden, dropout=DROPOUT):
        super().__init__()

        self.rad_encoder = nn.Sequential(
            nn.Linear(rad_dim, rad_hidden),
            nn.ReLU()
        )

        self.deep_encoder = nn.Sequential(
            nn.Linear(deep_dim, deep_hidden),
            nn.ReLU()
        )

        self.rad_gate = nn.Parameter(torch.full((rad_hidden,), -1.0))
        self.deep_gate = nn.Parameter(torch.full((deep_hidden,), 1.0))

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(rad_hidden + deep_hidden, 1)

    def forward(self, x_rad, x_deep):
        h_rad = self.rad_encoder(x_rad)
        h_deep = self.deep_encoder(x_deep)

        g_rad = torch.sigmoid(self.rad_gate).unsqueeze(0)
        g_deep = torch.sigmoid(self.deep_gate).unsqueeze(0)

        h_rad = self.dropout(h_rad * g_rad)
        h_deep = self.dropout(h_deep * g_deep)

        h = torch.cat([h_rad, h_deep], dim=1)
        out = self.classifier(h)
        return out.squeeze(1)


# ================= TRAIN / EVAL =================
def make_inner_split(y_outer_train, seed=SEED, val_frac=INNER_VAL_FRAC):
    y_outer_train = np.asarray(y_outer_train)

    class_counts = np.bincount(y_outer_train, minlength=2)
    if np.min(class_counts) < 2:
        n = len(y_outer_train)
        n_val = max(1, int(round(n * val_frac)))
        idx = np.arange(n)
        return idx[n_val:], idx[:n_val]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    tr_idx, es_idx = next(sss.split(np.zeros(len(y_outer_train)), y_outer_train))
    return tr_idx, es_idx


def train_one_fold(
    Xr_tr_raw, Xr_val_raw,
    Xd_tr_raw, Xd_val_raw,
    y_tr, y_val,
    rad_hidden, deep_hidden,
    fold_seed
):
    inner_tr_idx, inner_es_idx = make_inner_split(y_tr, seed=fold_seed, val_frac=INNER_VAL_FRAC)

    Xr_fit_raw, Xr_es_raw = Xr_tr_raw[inner_tr_idx], Xr_tr_raw[inner_es_idx]
    Xd_fit_raw, Xd_es_raw = Xd_tr_raw[inner_tr_idx], Xd_tr_raw[inner_es_idx]
    y_fit, y_es = y_tr[inner_tr_idx], y_tr[inner_es_idx]

    rad_proc = RadiomicsPreprocessor(max_features=MAX_RAD_FEATURES)
    deep_proc = DeepPreprocessor()

    Xr_fit = rad_proc.fit_transform(Xr_fit_raw, y_fit)
    Xr_es = rad_proc.transform(Xr_es_raw)
    Xr_val = rad_proc.transform(Xr_val_raw)

    Xd_fit = deep_proc.fit_transform(Xd_fit_raw)
    Xd_es = deep_proc.transform(Xd_es_raw)
    Xd_val = deep_proc.transform(Xd_val_raw)

    Xr_fit_t = torch.tensor(Xr_fit, dtype=torch.float32, device=DEVICE)
    Xr_es_t = torch.tensor(Xr_es, dtype=torch.float32, device=DEVICE)
    Xr_val_t = torch.tensor(Xr_val, dtype=torch.float32, device=DEVICE)

    Xd_fit_t = torch.tensor(Xd_fit, dtype=torch.float32, device=DEVICE)
    Xd_es_t = torch.tensor(Xd_es, dtype=torch.float32, device=DEVICE)
    Xd_val_t = torch.tensor(Xd_val, dtype=torch.float32, device=DEVICE)

    y_fit_t = torch.tensor(y_fit, dtype=torch.float32, device=DEVICE)

    model = FusionMLP(
        rad_dim=Xr_fit.shape[1],
        deep_dim=Xd_fit.shape[1],
        rad_hidden=rad_hidden,
        deep_hidden=deep_hidden,
        dropout=DROPOUT
    ).to(DEVICE)

    n_pos = max(int((y_fit == 1).sum()), 1)
    n_neg = max(int((y_fit == 0).sum()), 1)
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_state = None
    best_score = -np.inf
    best_epoch = 0
    wait = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()

        logits = model(Xr_fit_t, Xd_fit_t)
        loss = loss_fn(logits, y_fit_t)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            es_probs = torch.sigmoid(model(Xr_es_t, Xd_es_t)).cpu().numpy()

        es_auc = safe_auc(y_es, es_probs)
        if np.isnan(es_auc):
            es_pred_05 = (es_probs >= 0.5).astype(int)
            score = balanced_accuracy_score(y_es, es_pred_05)
        else:
            score = es_auc

        if score > best_score + MIN_DELTA:
            best_score = score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        es_probs = torch.sigmoid(model(Xr_es_t, Xd_es_t)).cpu().numpy()
        val_probs = torch.sigmoid(model(Xr_val_t, Xd_val_t)).cpu().numpy()

    best_thr, es_best_bal = tune_threshold(y_es, es_probs, grid=THRESH_GRID)
    val_preds = (val_probs >= best_thr).astype(int)

    fold_bal_acc = balanced_accuracy_score(y_val, val_preds)
    fold_auc = safe_auc(y_val, val_probs)
    fold_sen, fold_spe = compute_sen_spe(y_val, val_preds)

    fold_info = {
        "balanced accuracy": fold_bal_acc,
        "auc": fold_auc,
        "sen": fold_sen,
        "spe": fold_spe,
        "threshold": best_thr,
        "inner_best_bal_acc": es_best_bal,
        "best_epoch": best_epoch,
        "n_rad_selected": rad_proc.n_selected(),
        "n_deep_features_used": deep_proc.n_features_used()
    }

    return val_probs, val_preds, fold_info


def train_eval_fusion(X_rad, X_deep, y, rad_hidden, deep_hidden):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    all_probs = np.zeros(len(y), dtype=np.float32)
    all_preds = np.zeros(len(y), dtype=np.int32)
    fold_rows = []

    for fold_idx, (tr, val) in enumerate(skf.split(X_rad, y), start=1):
        Xr_tr_raw, Xr_val_raw = X_rad[tr], X_rad[val]
        Xd_tr_raw, Xd_val_raw = X_deep[tr], X_deep[val]
        y_tr, y_val = y[tr], y[val]

        val_probs, val_preds, fold_info = train_one_fold(
            Xr_tr_raw=Xr_tr_raw,
            Xr_val_raw=Xr_val_raw,
            Xd_tr_raw=Xd_tr_raw,
            Xd_val_raw=Xd_val_raw,
            y_tr=y_tr,
            y_val=y_val,
            rad_hidden=rad_hidden,
            deep_hidden=deep_hidden,
            fold_seed=SEED + fold_idx
        )

        all_probs[val] = val_probs
        all_preds[val] = val_preds

        fold_rows.append({
            "fold": fold_idx,
            "rad_hidden": rad_hidden,
            "deep_hidden": deep_hidden,
            **fold_info
        })

    overall_bal_acc = balanced_accuracy_score(y, all_preds)
    overall_auc = safe_auc(y, all_probs)
    overall_sen, overall_spe = compute_sen_spe(y, all_preds)

    return overall_bal_acc, overall_auc, overall_sen, overall_spe, all_probs, all_preds, pd.DataFrame(fold_rows)


# ================= PLOTTING =================
def modality_display_name(modality):
    if modality == "T1":
        return "T1"
    elif modality == "T2":
        return "T2"
    elif modality == "ADC":
        return "ADC"
    return modality


def save_combined_heatmap(results_df, metric="auc", modality_order=None):
    if modality_order is None:
        modality_order = ["T1", "T2", "ADC"]

    metric_label_map = {
        "auc": "AUROC",
        "balanced accuracy": "Balanced Accuracy",
        "sen": "Sensitivity",
        "spe": "Specificity",
    }
    metric_label = metric_label_map.get(metric, metric)

    fig, axes = plt.subplots(1, len(modality_order), figsize=(22, 6), constrained_layout=False)

    if len(modality_order) == 1:
        axes = [axes]

    panel_labels = ["(a)", "(b)", "(c)"]

    for idx, (ax, modality) in enumerate(zip(axes, modality_order)):
        sub = results_df[results_df["modality"] == modality].copy()

        if "T1" in modality:
            plot_modality = "T1w"
        elif "T2" in modality:
            plot_modality = "T2w"
        else:
            plot_modality = "DWI"

        panel_prefix = panel_labels[idx] if idx < len(panel_labels) else f"({chr(97 + idx)})"

        if sub.empty:
            ax.axis("off")
            ax.set_title(
                f"{panel_prefix} {modality_display_name(plot_modality)} - {metric_label}",
                fontsize=18,
                pad=12
            )
            continue

        pivot = sub.pivot_table(
            index="rad_hidden",
            columns="deep_hidden",
            values=metric,
            aggfunc="mean"
        )

        # enforce displayed order to match requested style
        pivot = pivot.reindex(index=[8, 16, 32], columns=[16, 32, 64, 128])

        im = ax.imshow(
        pivot.values,
        aspect="equal",
        interpolation="nearest",
        cmap="OrRd",
        vmin=0.5,
        vmax=0.8
    )

        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_xticklabels(pivot.columns, fontsize=15)
        ax.set_yticklabels(pivot.index, fontsize=15)

        ax.set_xlabel("Deep Hidden Layer Size", fontsize=17)
        ax.set_ylabel("Radiomics Hidden Layer Size", fontsize=17)

        ax.set_title(
            f"{panel_prefix} {modality_display_name(plot_modality)} - {metric_label}",
            fontsize=18,
            pad=12
        )

        from matplotlib import colors
        norm = colors.Normalize(vmin=0.0, vmax=1.0)

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    text_color = "white" if norm(val) > 0.8 else "black"

                    ax.text(
                        j, i, f"{val:.3f}",
                        ha="center", va="center",
                        fontsize=16,
                        color=text_color,
                        fontweight="bold"
                    )
                    # optional thin outline for even better readability
                    # txt.set_path_effects([
                    #     pe.withStroke(linewidth=1.2, foreground="black" if text_color == "white" else "white")
                    # ])

        cbar = fig.colorbar(im, ax=ax, shrink=0.92)
        cbar.ax.tick_params(labelsize=12)

    safe_metric_name = metric.replace(" ", "_")
    out_path = os.path.join(PLOT_DIR, f"combined_heatmap_{safe_metric_name}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

def main():
    print(f"Using device: {DEVICE}")

    labels_df = load_labels()

    print(f"DEV size from labels: {len(labels_df)}")
    print(labels_df["label"].value_counts(dropna=False))
    print()

    all_results = []
    all_fold_results = []

    for modality in MODALITIES:
        print("=" * 80)
        print(f"MODALITY: {modality}")

        rad_df = load_radiomics(modality)
        deep_df = load_deep(modality)

        merged = labels_df.merge(rad_df, on="id", how="left")
        merged = merged.merge(deep_df, on="id", how="left")

        rad_cols = [
            c for c in merged.columns
            if c not in ["id", "label", "site"] and not c.startswith("medsiglip_")
        ]
        deep_cols = [c for c in merged.columns if c.startswith("medsiglip_")]

        if len(rad_cols) == 0:
            print(f"Skipping {modality}: no radiomics features found")
            continue
        if len(deep_cols) == 0:
            print(f"Skipping {modality}: no deep features found")
            continue

        merged[rad_cols] = merged[rad_cols].apply(pd.to_numeric, errors="coerce")
        merged[deep_cols] = merged[deep_cols].apply(pd.to_numeric, errors="coerce")

        X_rad = merged[rad_cols].to_numpy(dtype=np.float32)
        X_deep = merged[deep_cols].to_numpy(dtype=np.float32)
        y = merged["label"].to_numpy(dtype=int)

        X_rad[np.isinf(X_rad)] = np.nan
        X_deep[np.isinf(X_deep)] = np.nan

        rad_missing_all = np.isnan(X_rad).all(axis=1)
        deep_missing_all = np.isnan(X_deep).all(axis=1)

        keep_mask = ~(rad_missing_all & deep_missing_all)

        dropped_both = int((~keep_mask).sum())
        kept_only_rad = int((~rad_missing_all & deep_missing_all).sum())
        kept_only_deep = int((rad_missing_all & ~deep_missing_all).sum())
        kept_both_present = int((~rad_missing_all & ~deep_missing_all).sum())

        X_rad = X_rad[keep_mask]
        X_deep = X_deep[keep_mask]
        y = y[keep_mask]

        rad_missing_ind = rad_missing_all[keep_mask].astype(np.float32).reshape(-1, 1)
        deep_missing_ind = deep_missing_all[keep_mask].astype(np.float32).reshape(-1, 1)

        X_rad = np.concatenate([X_rad, rad_missing_ind], axis=1)
        X_deep = np.concatenate([X_deep, deep_missing_ind], axis=1)

        print(f"Labels rows:          {len(labels_df)}")
        print(f"After merge rows:     {len(merged)}")
        print(f"Dropped both missing: {dropped_both}")
        print(f"Kept only rad:        {kept_only_rad}")
        print(f"Kept only deep:       {kept_only_deep}")
        print(f"Kept both present:    {kept_both_present}")
        print(f"Final samples:        {len(y)}")
        print(f"Radiomics shape:      {X_rad.shape}")
        print(f"Deep shape:           {X_deep.shape}")
        print()

        if len(y) == 0:
            print(f"Skipping {modality}: no usable samples after filtering")
            continue

        for rad_h in RAD_HIDDEN_SIZES:
            for deep_h in DEEP_HIDDEN_SIZES:
                bal_acc, auc, sen, spe, probs, preds, fold_df = train_eval_fusion(
                    X_rad=X_rad,
                    X_deep=X_deep,
                    y=y,
                    rad_hidden=rad_h,
                    deep_hidden=deep_h
                )

                mean_rad_selected = fold_df["n_rad_selected"].mean()
                mean_thr = fold_df["threshold"].mean()
                mean_best_epoch = fold_df["best_epoch"].mean()

                print(
                    f"Fusion ({X_rad.shape[1]}→{rad_h}) + ({X_deep.shape[1]}→{deep_h}) → 1 | "
                    f"Balanced Acc = {bal_acc:.4f}, AUC = {auc:.4f}, "
                    f"SEN = {sen:.4f}, SPE = {spe:.4f}, "
                    f"Mean selected rad = {mean_rad_selected:.1f}, Mean thr = {mean_thr:.2f}"
                )

                all_results.append({
                    "modality": modality,
                    "n_samples": len(y),
                    "n_rad_features": X_rad.shape[1],
                    "n_deep_features": X_deep.shape[1],
                    "rad_hidden": rad_h,
                    "deep_hidden": deep_h,
                    "balanced accuracy": bal_acc,
                    "auc": auc,
                    "sen": sen,
                    "spe": spe,
                    "mean_rad_selected": mean_rad_selected,
                    "mean_threshold": mean_thr,
                    "mean_best_epoch": mean_best_epoch,
                    "dropped_both_missing": dropped_both,
                    "kept_only_rad": kept_only_rad,
                    "kept_only_deep": kept_only_deep,
                    "kept_both_present": kept_both_present
                })

                fold_df["modality"] = modality
                fold_df["n_samples"] = len(y)
                fold_df["n_rad_features"] = X_rad.shape[1]
                fold_df["n_deep_features"] = X_deep.shape[1]
                fold_df["dropped_both_missing"] = dropped_both
                fold_df["kept_only_rad"] = kept_only_rad
                fold_df["kept_only_deep"] = kept_only_deep
                fold_df["kept_both_present"] = kept_both_present
                all_fold_results.append(fold_df)

        print()

    if len(all_results) == 0:
        print("No results generated.")
        return

    results_df = pd.DataFrame(all_results)
    fold_results_df = pd.concat(all_fold_results, ignore_index=True)

    results_csv = os.path.join(OUTPUT_DIR, "fusion_summary.csv")
    fold_csv = os.path.join(OUTPUT_DIR, "fusion_fold_results.csv")
    results_xlsx = os.path.join(OUTPUT_DIR, "fusion_summary.xlsx")
    fold_xlsx = os.path.join(OUTPUT_DIR, "fusion_fold_results.xlsx")

    results_df.to_csv(results_csv, index=False)
    fold_results_df.to_csv(fold_csv, index=False)
    results_df.to_excel(results_xlsx, index=False)
    fold_results_df.to_excel(fold_xlsx, index=False)

    best_by_modality = (
        results_df.sort_values(
            ["modality", "balanced accuracy", "auc"],
            ascending=[True, False, False]
        )
        .groupby("modality", as_index=False)
        .first()
    )

    best_csv = os.path.join(OUTPUT_DIR, "fusion_best_by_modality.csv")
    best_xlsx = os.path.join(OUTPUT_DIR, "fusion_best_by_modality.xlsx")
    best_by_modality.to_csv(best_csv, index=False)
    best_by_modality.to_excel(best_xlsx, index=False)

    overall_best = (
        results_df.sort_values(["balanced accuracy", "auc"], ascending=False)
        .iloc[0]
        .to_dict()
    )
    overall_best_json = os.path.join(OUTPUT_DIR, "fusion_overall_best.json")
    with open(overall_best_json, "w") as f:
        json.dump(overall_best, f, indent=2)

    # two combined figures only
    save_combined_heatmap(results_df, metric="auc", modality_order=["T1", "T2", "ADC"])
    save_combined_heatmap(results_df, metric="balanced accuracy", modality_order=["T1", "T2", "ADC"])

    print("=" * 80)
    print("DONE")
    print(f"Saved summary CSV:         {results_csv}")
    print(f"Saved fold CSV:            {fold_csv}")
    print(f"Saved summary Excel:       {results_xlsx}")
    print(f"Saved fold Excel:          {fold_xlsx}")
    print(f"Saved best per mod CSV:    {best_csv}")
    print(f"Saved best per mod Excel:  {best_xlsx}")
    print(f"Saved overall best JSON:   {overall_best_json}")
    print(f"Saved plots in:            {PLOT_DIR}")
    print(f"  - {os.path.join(PLOT_DIR, 'combined_heatmap_auc.png')}")
    print(f"  - {os.path.join(PLOT_DIR, 'combined_heatmap_balanced_accuracy.png')}")
    print()

    print("Best by modality:")
    print(
        best_by_modality[
            [
                "modality",
                "n_samples",
                "n_rad_features",
                "n_deep_features",
                "rad_hidden",
                "deep_hidden",
                "balanced accuracy",
                "auc",
                "sen",
                "spe",
                "mean_rad_selected",
                "mean_threshold",
                "mean_best_epoch",
                "dropped_both_missing",
                "kept_only_rad",
                "kept_only_deep",
                "kept_both_present"
            ]
        ].to_string(index=False)
    )
    print()

    print("Overall best:")
    print(json.dumps(overall_best, indent=2))


if __name__ == "__main__":
    main()

Using device: cuda
DEV size from labels: 438
label
0    220
1    218
Name: count, dtype: int64

MODALITY: T1
Labels rows:          438
After merge rows:     438
Dropped both missing: 141
Kept only rad:        0
Kept only deep:       39
Kept both present:    258
Final samples:        297
Radiomics shape:      (297, 201)
Deep shape:           (297, 1153)

Fusion (201→32) + (1153→128) → 1 | Balanced Acc = 0.6417, AUC = 0.6883, SEN = 0.5616, SPE = 0.7219, Mean selected rad = 16.0, Mean thr = 0.53
Fusion (201→32) + (1153→64) → 1 | Balanced Acc = 0.6220, AUC = 0.6646, SEN = 0.5685, SPE = 0.6755, Mean selected rad = 16.0, Mean thr = 0.52
Fusion (201→32) + (1153→32) → 1 | Balanced Acc = 0.6031, AUC = 0.6554, SEN = 0.6301, SPE = 0.5762, Mean selected rad = 16.0, Mean thr = 0.50
Fusion (201→32) + (1153→16) → 1 | Balanced Acc = 0.6164, AUC = 0.6241, SEN = 0.6301, SPE = 0.6026, Mean selected rad = 16.0, Mean thr = 0.48
Fusion (201→16) + (1153→128) → 1 | Balanced Acc = 0.6211, AUC = 0.6580, SEN = 0